In [13]:
import os
import sys
import numpy as np
import csv
import time
from itertools import product

project_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_dir)

from src.recommender_system import *
from src.core import *
from src.tests import *

In [14]:
# region, level, year = get_region("taiwan") # world, india, usa, taiwan

In [15]:
# data, params, trace = build_cartogram(
#     region = region,
#     level = level,
#     year = year,
#     cartogram_type = "contiguous-without-postprocess",  # contiguous; noncontiguous; dorling; demers; contiguous-without-postprocess
#     size_accuracy = 1.0,
#     shape_preservation = 0.0,
#     position_accuracy = 0.5,
#     neighborhoods_kept = 1.0,
#     seamlessness = 0.0,
#     robustness = 1.0,
#     overrides = None,
# )

In [16]:
# plot_polys_data(data,  title=f"Recommmended cartogram for: {region}")

In [17]:
# quality = quality_criteria(data, cartogram_key="polygon")
# for key in quality.keys():
#     print(f"{key}: {quality[key]}")

In [ ]:
# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
# Regions to test (names must match the keys used by get_region)
REGIONS = ["world"]#"usa", "india", "taiwan", "netherlands", "germany"]

# Cartogram type to use (from recommender_system's BASE_PRESETS)
CARTOGRAM_TYPE = "contiguous"   # or "contiguous", "dorling", "demers", "noncontiguous"

# Sliders to vary (keys used in build_cartogram)
SLIDERS = ["size_accuracy", "shape_preservation", "position_accuracy",
           "neighborhoods_kept", "seamlessness", "robustness"]

# Values to test for each slider (besides the base value 0.5)
SLIDER_VALUES = [0.0, 1.0]

# Output CSV file
OUTPUT_CSV = "recommender_test_results_contiguous_world.csv"

# ----------------------------------------------------------------------
# Helper: run one experiment
# ----------------------------------------------------------------------
def run_experiment(region, level, year, cartogram_type, slider_overrides):
    """
    Build a cartogram with the given slider settings and return quality metrics.
    slider_overrides: dict {slider_name: value} to override the default 0.5.
    """
    # Build default slider dict (all 0.5)
    slider_kwargs = {s: 0.5 for s in SLIDERS}
    # Apply overrides
    slider_kwargs.update(slider_overrides)

    try:
        data, params, trace = build_cartogram(
            region=region,
            level=level,
            year=year,
            cartogram_type=cartogram_type,
            **slider_kwargs,
            verbose=False   # suppress print output
        )
        # Compute quality metrics; use cartogram_key="polygon" because build_cartogram stores output under "polygon"
        quality = quality_criteria(data, cartogram_key="polygon")
        return quality
    except Exception as e:
        print(f"    ERROR: {e}")
        return None

# ----------------------------------------------------------------------
# Main loop
# ----------------------------------------------------------------------
def main():
    all_rows = []

    # Prepare CSV header: region, slider, value, plus all metric names
    # We need to know the metric names from a dummy run
    dummy_region, dummy_level, dummy_year = get_region("usa")
    dummy_quality = run_experiment(dummy_region, dummy_level, dummy_year,
                                   CARTOGRAM_TYPE, {})
    if dummy_quality is None:
        print("Could not obtain dummy quality metrics. Exiting.")
        sys.exit(1)

    metric_names = list(dummy_quality.keys())
    header = ["region", "slider", "value"] + metric_names

    # Loop over regions
    for region_name in REGIONS:
        print(f"\nProcessing region: {region_name}")
        region, level, year = get_region(region_name)

        # 1. Base run (all sliders at 0.5)
        print("  Base run (all sliders 0.5)")
        base_quality = run_experiment(region, level, year, CARTOGRAM_TYPE, {})
        if base_quality is not None:
            row = {"region": region_name, "slider": "base", "value": 0.5, **base_quality}
            all_rows.append(row)

        # 2. For each slider, test values 0.0 and 1.0
        for slider in SLIDERS:
            for val in SLIDER_VALUES:
                print(f"  Slider: {slider} = {val}")
                overrides = {slider: val}
                quality = run_experiment(region, level, year, CARTOGRAM_TYPE, overrides)
                if quality is not None:
                    row = {"region": region_name, "slider": slider, "value": val, **quality}
                    all_rows.append(row)

    # Write CSV
    if all_rows:
        with open(OUTPUT_CSV, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=header)
            writer.writeheader()
            for row in all_rows:
                # Ensure all metric keys are present (they should be)
                writer.writerow(row)
        print(f"\nResults written to {OUTPUT_CSV}")
    else:
        print("No results collected.")

if __name__ == "__main__":
    main()

Matched name column: 'Entity', value column: 'electoral_college'
Number of unmatched regions for year 2016: 0
Contiguous mode 'contiguous2': 2055 shared-vertex penalty terms added.


/home/kuenem/Documents/development/lectures/Master Thesis/Cartogram Framework/src/core/CartogramFramework_data.py:923: UserWarning: Objective contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  prob = cp.Problem(objective, constraints)
/home/kuenem/Documents/development/lectures/Master Thesis/Cartogram Framework/src/core/CartogramFramework_data.py:924: UserWarning: Objective contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  prob.solve(solver=cp.CLARABEL, verbose=False)


Status : optimal
Obj val: 7.32028e+06
Post-processing: equalizing region areas on the raster grid (max_passes=30, tolerance=0.02)

Processing region: world
  Base run (all sliders 0.5)
Matched name column: 'name', value column: 'Data'
Number of unmatched regions for year 2020: 0


/home/kuenem/miniconda3/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


Contiguous mode 'contiguous2': 19262 shared-vertex penalty terms added.


/home/kuenem/Documents/development/lectures/Master Thesis/Cartogram Framework/src/core/CartogramFramework_data.py:923: UserWarning: Objective contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  prob = cp.Problem(objective, constraints)
/home/kuenem/Documents/development/lectures/Master Thesis/Cartogram Framework/src/core/CartogramFramework_data.py:924: UserWarning: Objective contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  prob.solve(solver=cp.CLARABEL, verbose=False)


Status : infeasible
Obj val: inf
  Note: after Fix 1, every constraint is a box bound or a one-sided slack (never a hard equality), so this status should only occur from solver numerical/timeout issues, not structural infeasibility. If it persists, try:
    • Lowering lambda_order / lambda_contiguous / lambda_mean_scale (very large weights can make the QP numerically stiff)
    • Loosening t_min/t_max
    ERROR: 'new_area'
  Slider: size_accuracy = 0.0
Matched name column: 'name', value column: 'Data'
Number of unmatched regions for year 2020: 0
Contiguous mode 'contiguous2': 19262 shared-vertex penalty terms added.


/home/kuenem/Documents/development/lectures/Master Thesis/Cartogram Framework/src/core/CartogramFramework_data.py:923: UserWarning: Objective contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  prob = cp.Problem(objective, constraints)
/home/kuenem/Documents/development/lectures/Master Thesis/Cartogram Framework/src/core/CartogramFramework_data.py:924: UserWarning: Objective contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  prob.solve(solver=cp.CLARABEL, verbose=False)


Status : infeasible
Obj val: inf
  Note: after Fix 1, every constraint is a box bound or a one-sided slack (never a hard equality), so this status should only occur from solver numerical/timeout issues, not structural infeasibility. If it persists, try:
    • Lowering lambda_order / lambda_contiguous / lambda_mean_scale (very large weights can make the QP numerically stiff)
    • Loosening t_min/t_max
    ERROR: 'new_area'
  Slider: size_accuracy = 1.0
Matched name column: 'name', value column: 'Data'
Number of unmatched regions for year 2020: 0
Contiguous mode 'contiguous2': 19262 shared-vertex penalty terms added.


/home/kuenem/Documents/development/lectures/Master Thesis/Cartogram Framework/src/core/CartogramFramework_data.py:923: UserWarning: Objective contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  prob = cp.Problem(objective, constraints)
/home/kuenem/Documents/development/lectures/Master Thesis/Cartogram Framework/src/core/CartogramFramework_data.py:924: UserWarning: Objective contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  prob.solve(solver=cp.CLARABEL, verbose=False)
